# 2 – Preprocesado de Datos

En este notebook preparamos el dataset para los modelos de ML.  
Las decisiones metodológicas principales son:

1. Codificación de variables categóricas
2. Creación de versiones del dataset para clasificación y regresión
3. Separación del conjunto de entrenamiento y test
4. Escalado de variables numéricas


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('rendimiento_estudiantes.csv', sep=';')
print(f"Dataset cargado: {df.shape}")
df.head(3)


Dataset cargado: (4424, 37)


,estado_civil,modo_solicitud,orden_solicitud,curso,asistencia_diurna_vespertina,cualificacion_previa,nota_cualificacion_previa,nacionalidad,cualificacion_madre,cualificacion_padre,...,asignaturas_2sem_convalidadas,asignaturas_2sem_matriculadas,asignaturas_2sem_evaluadas,asignaturas_2sem_aprobadas,nota_media_2sem,asignaturas_2sem_sin_evaluacion,tasa_desempleo,tasa_inflacion,pib,objetivo
0,soltero/a,2a_fase_contingente_general,5,animacion_y_diseno_multimedia,diurna,educacion_secundaria,122.0,portuguesa,educacion_basica_3er_ciclo_o_equivalente,otro_11o_ano,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,abandono
1,soltero/a,estudiante_internacional_licenciatura,1,turismo,diurna,educacion_secundaria,160.0,portuguesa,educacion_secundaria_12o_ano_o_equivalente,educacion_superior_grado,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,graduado
2,soltero/a,1a_fase_contingente_general,5,diseno_de_comunicacion,diurna,educacion_secundaria,122.0,portuguesa,educacion_basica_1er_ciclo_o_equivalente,educacion_basica_1er_ciclo_o_equivalente,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,abandono


## 2.1 Codificación de la variable objetivo

In [2]:
# Codificamos la variable objetivo para clasificación
le_objetivo = LabelEncoder()
df['objetivo_enc'] = le_objetivo.fit_transform(df['objetivo'])
df.drop(columns='objetivo',inplace=True)
print("Codificación de clases:")
for cls, enc in zip(le_objetivo.classes_, range(len(le_objetivo.classes_))):
    print(f"· {cls} -> {enc}")
print()
print(df['objetivo_enc'].value_counts())


Codificación de clases:
· abandono -> 0
· graduado -> 1
· matriculado -> 2

objetivo_enc
1    2209
0    1421
2     794
Name: count, dtype: int64


## 2.2 Codificación de variables categóricas

In [3]:
# Identificamos las columnas categóricas (excepto objetivo)
cols_cat = df.select_dtypes(include='object').columns.tolist()
cols_num = df.select_dtypes(include='number').columns.tolist()
if 'objetivo_enc' in cols_cat:
    cols_cat.remove('objetivo_enc') 
elif 'objetivo_enc' in cols_num:
   cols_num.remove('objetivo_enc')


print(f"Variables categóricas a codificar ({len(cols_cat)}):")
for c in cols_cat:
    n_unique = df[c].nunique()
    print(f"· {c}: {n_unique} valores únicos")


Variables categóricas a codificar (17):
· estado_civil: 6 valores únicos
· modo_solicitud: 18 valores únicos
· curso: 17 valores únicos
· asistencia_diurna_vespertina: 2 valores únicos
· cualificacion_previa: 17 valores únicos
· nacionalidad: 21 valores únicos
· cualificacion_madre: 29 valores únicos
· cualificacion_padre: 34 valores únicos
· ocupacion_madre: 32 valores únicos
· ocupacion_padre: 46 valores únicos
· desplazado: 2 valores únicos
· necesidades_educativas_especiales: 2 valores únicos
· deudor: 2 valores únicos
· matricula_al_dia: 2 valores únicos
· genero: 2 valores únicos
· becado: 2 valores únicos
· internacional: 2 valores únicos


In [4]:
df_enc=df.copy()

# Variables binarias tienen exactamente 2 categorías, aplicamos LabelEncoder
cols_bin = [
    'asistencia_diurna_vespertina',   # diurna / vespertina
    'desplazado',                      # si / no
    'necesidades_educativas_especiales',# si / no
    'deudor',                          # si / no
    'matricula_al_dia',                # si / no
    'genero',                          # hombre / mujer
    'becado',                          # si / no
    'internacional',                   # si / no
]

le_bin = {}  # columna: LabelEncoder ajustado

for col in cols_bin:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df[col])
    le_bin[col] = le
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"{col}: {mapping}")
df_enc.head(10)


asistencia_diurna_vespertina: {'diurna': np.int64(0), 'vespertina': np.int64(1)}
desplazado: {'no': np.int64(0), 'si': np.int64(1)}
necesidades_educativas_especiales: {'no': np.int64(0), 'si': np.int64(1)}
deudor: {'no': np.int64(0), 'si': np.int64(1)}
matricula_al_dia: {'no': np.int64(0), 'si': np.int64(1)}
genero: {'hombre': np.int64(0), 'mujer': np.int64(1)}
becado: {'no': np.int64(0), 'si': np.int64(1)}
internacional: {'no': np.int64(0), 'si': np.int64(1)}


,estado_civil,modo_solicitud,orden_solicitud,curso,asistencia_diurna_vespertina,cualificacion_previa,nota_cualificacion_previa,nacionalidad,cualificacion_madre,cualificacion_padre,...,asignaturas_2sem_convalidadas,asignaturas_2sem_matriculadas,asignaturas_2sem_evaluadas,asignaturas_2sem_aprobadas,nota_media_2sem,asignaturas_2sem_sin_evaluacion,tasa_desempleo,tasa_inflacion,pib,objetivo_enc
0,soltero/a,2a_fase_contingente_general,5,animacion_y_diseno_multimedia,0,educacion_secundaria,122.0,portuguesa,educacion_basica_3er_ciclo_o_equivalente,otro_11o_ano,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,0
1,soltero/a,estudiante_internacional_licenciatura,1,turismo,0,educacion_secundaria,160.0,portuguesa,educacion_secundaria_12o_ano_o_equivalente,educacion_superior_grado,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,1
2,soltero/a,1a_fase_contingente_general,5,diseno_de_comunicacion,0,educacion_secundaria,122.0,portuguesa,educacion_basica_1er_ciclo_o_equivalente,educacion_basica_1er_ciclo_o_equivalente,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,0
3,soltero/a,2a_fase_contingente_general,2,periodismo_y_comunicacion,0,educacion_secundaria,122.0,portuguesa,educacion_basica_2o_ciclo_o_equivalente,educacion_basica_1er_ciclo_o_equivalente,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,1
4,casado/a,mayor_de_23_anos,1,servicio_social_nocturno,1,educacion_secundaria,100.0,portuguesa,educacion_basica_1er_ciclo_o_equivalente,educacion_basica_2o_ciclo_o_equivalente,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,1
5,casado/a,mayor_de_23_anos,1,gestion_nocturno,1,educacion_basica_3er_ciclo_o_equivalente,133.1,portuguesa,educacion_basica_1er_ciclo_o_equivalente,educacion_basica_1er_ciclo_o_equivalente,...,0,5,17,5,11.500000,5,16.2,0.3,-0.92,1
6,soltero/a,1a_fase_contingente_general,1,enfermeria,0,educacion_secundaria,142.0,portuguesa,educacion_basica_3er_ciclo_o_equivalente,educacion_basica_2o_ciclo_o_equivalente,...,0,8,8,8,14.345000,0,15.5,2.8,-4.06,1
7,soltero/a,3a_fase_contingente_general,4,turismo,0,educacion_secundaria,119.0,portuguesa,educacion_basica_1er_ciclo_o_equivalente,educacion_basica_1er_ciclo_o_equivalente,...,0,5,5,0,0.000000,0,15.5,2.8,-4.06,0
8,soltero/a,1a_fase_contingente_general,3,servicio_social,0,educacion_secundaria,137.0,rumana,educacion_secundaria_12o_ano_o_equivalente,educacion_secundaria_12o_ano_o_equivalente,...,0,6,7,6,14.142857,0,16.2,0.3,-0.92,1
9,soltero/a,1a_fase_contingente_general,1,servicio_social,0,educacion_secundaria,138.0,portuguesa,educacion_secundaria_12o_ano_o_equivalente,educacion_basica_3er_ciclo_o_equivalente,...,0,6,14,2,13.500000,0,8.9,1.4,3.51,0


Codificación de variables nominales con OneHotEncoder. Nos va a generar muchas columnas, lo cual es una limitación de este modelo.

Usamos `drop='first'` para eliminar una categoría por variable y ahorrar una variable extra.  
Con `sparse_output=False` obtenemos directamente un array denso y con `handle_unknown='ignore'` el encoder no falla si en test aparece una categoría no vista en entrenamiento (la trata como la categoría de referencia).

In [5]:
# Variables nominales son varias categorías sin orden natural, aplicamos OneHotEncoder
cols_ohe = [
    'estado_civil',        # 6 categorías
    'modo_solicitud',      # 18 categorías
    'curso',               # 17 categorías
    'cualificacion_previa',# 17 categorías (vía de acceso, sin orden claro)
    'nacionalidad',        # 21 categorías
    'cualificacion_madre', # 29 categorías
    'cualificacion_padre', # 34 categorías
    'ocupacion_madre',     # 32 categorías
    'ocupacion_padre',     # 46 categorías
]
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
ohe.fit(df[cols_ohe])

# Nombres de las columnas generadas
nombre_features_ohe = ohe.get_feature_names_out(cols_ohe)
print(f"Columnas OHE generadas: {len(nombre_features_ohe)}")

# Resumen por variable
for col, cats in zip(cols_ohe, ohe.categories_):
    print(f"  {col}: {len(cats)} categorías lleva {len(cats)-1} columnas OHE con referencia: {cats[0]}")

# Transformación OHE
ohe_array = ohe.transform(df[cols_ohe])
df_ohe = pd.DataFrame(ohe_array, columns=nombre_features_ohe, index=df.index)

df_enc = df_enc.drop(columns=cols_ohe)
df_enc = pd.concat([df_enc, df_ohe], axis=1)
df_enc.head(10)

Columnas OHE generadas: 211
  estado_civil: 6 categorías lleva 5 columnas OHE con referencia: casado/a
  modo_solicitud: 18 categorías lleva 17 columnas OHE con referencia: 1a_fase_contingente_especial_azores
  curso: 17 categorías lleva 16 columnas OHE con referencia: agronomia
  cualificacion_previa: 17 categorías lleva 16 columnas OHE con referencia: 10o_ano
  nacionalidad: 21 categorías lleva 20 columnas OHE con referencia: alemana
  cualificacion_madre: 29 categorías lleva 28 columnas OHE con referencia: 10o_ano
  cualificacion_padre: 34 categorías lleva 33 columnas OHE con referencia: 10o_ano
  ocupacion_madre: 32 categorías lleva 31 columnas OHE con referencia: agricultoras_y_trabajadoras_cualificadas_de_agricultura_pesca_y_silvicultura
  ocupacion_padre: 46 categorías lleva 45 columnas OHE con referencia: agricultores_ganaderos_pescadores_cazadores_y_recolectores_de_subsistencia


,orden_solicitud,asistencia_diurna_vespertina,nota_cualificacion_previa,nota_admision,desplazado,necesidades_educativas_especiales,deudor,matricula_al_dia,genero,becado,...,ocupacion_padre_trabajadores_cualificados_de_la_construccion_excepto_electricistas,ocupacion_padre_trabajadores_cualificados_de_metalurgia_metalmecanica_y_similares,ocupacion_padre_trabajadores_de_industrias_alimentarias_madera_confeccion_y_otras_industrias_y_artes,ocupacion_padre_trabajadores_de_servicios_personales,ocupacion_padre_trabajadores_de_servicios_personales_seguridad_y_vendedores,ocupacion_padre_trabajadores_no_cualificados,ocupacion_padre_trabajadores_no_cualificados_de_agricultura_ganaderia_pesca_y_silvicultura,ocupacion_padre_trabajadores_no_cualificados_de_industria_extractiva_construccion_manufactura_y_transporte,ocupacion_padre_vendedores,ocupacion_padre_vendedores_ambulantes_y_prestadores_de_servicios_callejeros
0,5,0,122.0,127.3,1,0,0,1,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,1,0,160.0,142.5,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,5,0,122.0,124.8,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,2,0,122.0,119.6,1,0,0,1,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,1,100.0,141.5,0,0,0,1,1,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5,1,1,133.1,114.8,0,0,1,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,1,0,142.0,128.4,1,0,0,1,1,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,4,0,119.0,113.1,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
8,3,0,137.0,129.3,0,0,0,1,1,1,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
9,1,0,138.0,123.0,1,0,1,0,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 2.3 Dataset para Tarea 1 – Clasificación

In [6]:
# Para clasificación usamos todas las variables disponibles (incluyendo variables del 2do semestre ya que predecimos el estado final)
features_clf = [c for c in df_enc.columns if c != 'objetivo_enc'] # todas las features
target_clf = 'objetivo_enc'

X_clf = df_enc[features_clf]
y_clf = df_enc[target_clf]

print(f"Features clasificación: {len(features_clf)}")
print(f"Target: {target_clf}")

# Train-test split estratificado (para mantener proporción de clases) con shuffle
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf, shuffle=True)

print(f"\nTrain: {X_train_clf.shape[0]} muestras")
print(f"Test:  {X_test_clf.shape[0]} muestras")

# Distribución de clases en train
print(f"\nDistribución en train:")
dist_train = y_train_clf.value_counts(normalize=True) * 100

print("\nDistribución en train:")
for v, pct in dist_train.items():
    print(f"  {le_objetivo.classes_[v]}: {pct:.1f}%")


Features clasificación: 238
Target: objetivo_enc

Train: 3539 muestras
Test:  885 muestras

Distribución en train:

Distribución en train:
  graduado: 49.9%
  abandono: 32.1%
  matriculado: 17.9%


In [7]:
# Escalamos para los modelos que lo necesiten (SVM, regresión logística, etc.)
scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)

print("Escalado completado.")
print(f"Media de X_train_scaled (debe ser cercano a 0): {X_train_clf_scaled.mean():.4f}")
print(f"Std de X_train_scaled (debe ser cercano a 1):   {X_train_clf_scaled.std()}")


Escalado completado.
Media de X_train_scaled (debe ser cercano a 0): -0.0000
Std de X_train_scaled (debe ser cercano a 1):   0.976617382911569


In [8]:
# Calculamos las desviaciones por columna
stds_por_columna = X_train_clf_scaled.std(axis=0)

# Buscamos cuáles tienen una desviación de 0
columnas_constantes = np.where(stds_por_columna == 0)[0]

if len(columnas_constantes) > 0:
    print(f"¡Cuidado! Hay varianza 0 en las columnas: ")
    # Las que salgan son fruto del OneHotEncoding, son 0 todos los valores que toman en realidad
    print(X_train_clf.columns[columnas_constantes])


¡Cuidado! Hay varianza 0 en las columnas: 
Index(['nacionalidad_colombiana', 'nacionalidad_lituana',
       'cualificacion_madre_2o_ciclo_del_bachillerato_general',
       'cualificacion_madre_curso_tecnico_profesional',
       'cualificacion_padre_2o_ano_complementario_de_secundaria',
       'cualificacion_padre_contabilidad_y_administracion_complementaria',
       'ocupacion_padre_agricultores_orientados_al_mercado_y_trabajadores_agropecuarios_cualificados',
       'ocupacion_padre_directores_de_hoteleria_restauracion_comercio_y_otros_servicios',
       'ocupacion_padre_directores_de_servicios_administrativos_y_comerciales',
       'ocupacion_padre_tecnicos_y_profesiones_intermedias_de_ciencia_e_ingenieria',
       'ocupacion_padre_vendedores_ambulantes_y_prestadores_de_servicios_callejeros'],
      dtype='object')


In [9]:
# Para solucionarlo:
# 1. Extraemos los nombres exactos de las columnas con varianza cero
columnas_a_eliminar = X_train_clf.columns[columnas_constantes]

# 2. Las eliminamos tanto del conjunto de entrenamiento como del de prueba
X_train_clf = X_train_clf.drop(columns=columnas_a_eliminar)
# Usamos errors='ignore' por si alguna de esas columnas no estuviera en el test
X_test_clf = X_test_clf.drop(columns=columnas_a_eliminar, errors='ignore') 

print(f"Se han eliminado {len(columnas_a_eliminar)} columnas constantes.")

# 3. Volvemos a instanciar y aplicar el escalador sobre los datos limpios
scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)

# 4. Comprobamos de nuevo
print(f"\nRe-escalado completado.")
print(f"Media de X_train_scaled: {X_train_clf_scaled.mean():.4f}")
print(f"Std de X_train_scaled:   {X_train_clf_scaled.std():.4f}")

#Las volvemos a guardar
features_clf = X_train_clf.columns.tolist()
print(f"Features finales para clasificación: {len(features_clf)}")

Se han eliminado 11 columnas constantes.



Re-escalado completado.
Media de X_train_scaled: -0.0000
Std de X_train_scaled:   1.0000
Features finales para clasificación: 227


Ahora ya vemos que es correcto el escalado para clasificacion

## 2.4 Dataset para Tarea 2 – Regresión

In [10]:
# Para regresión predecimos nota_media_2sem
# Debemos EXCLUIR variables del 2do semestre que son:
# - asignaturas_2sem_convalidadas, 
# - asignaturas_2sem_matriculadas, 
# - asignaturas_2sem_evaluadas,
# - asignaturas_2sem_aprobadas, 
# - asignaturas_2sem_sin_evaluacion
# - nota_media_2sem (el propio target)
# También excluimos la variable objetivo (no disponible en momento de predicción)

vars_leakage_2sem = [
    'asignaturas_2sem_convalidadas',
    'asignaturas_2sem_matriculadas',
    'asignaturas_2sem_evaluadas',
    'asignaturas_2sem_aprobadas',
    'asignaturas_2sem_sin_evaluacion',
    'nota_media_2sem',      # el target
    'objetivo_enc'          # no disponible en tiempo real
]

target_reg = 'nota_media_2sem'

# Seleccionamos features validas
features_reg = [c for c in df_enc.columns if c not in vars_leakage_2sem]
print(f"Features regresión (validas para pred): {len(features_reg)}")
print(f"Variables excluidas: {vars_leakage_2sem}")

X_reg = df_enc[features_reg]
y_reg = df_enc[target_reg]

# Solo usamos estudiantes con nota > 0 en el 2do semestre
# (los que tienen 0 suelen ser los que abandonaron sin llegar al 2do semestre)
mask_valido = y_reg > 0
X_reg = X_reg[mask_valido]
y_reg = y_reg[mask_valido]
print(f"\nEstudiantes con nota_media_2sem > 0: {len(y_reg)}")
print(f"(Se descartan {(~mask_valido).sum()} estudiantes con nota=0, probablemente abandonos tempranos)")


Features regresión (validas para pred): 232
Variables excluidas: ['asignaturas_2sem_convalidadas', 'asignaturas_2sem_matriculadas', 'asignaturas_2sem_evaluadas', 'asignaturas_2sem_aprobadas', 'asignaturas_2sem_sin_evaluacion', 'nota_media_2sem', 'objetivo_enc']

Estudiantes con nota_media_2sem > 0: 3554
(Se descartan 870 estudiantes con nota=0, probablemente abandonos tempranos)


In [11]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42, shuffle=True)

scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

print(f"\nEscalado completado.")
print(f"Media de X_train_scaled: {X_train_reg_scaled.mean():.4f}")
print(f"Std de X_train_scaled:   {X_train_reg_scaled.std():.4f}")


Escalado completado.
Media de X_train_scaled: 0.0000
Std de X_train_scaled:   0.9469


In [12]:
# Calculamos las desviaciones por columna
stds_por_columna = X_train_reg_scaled.std(axis=0)

# Buscamos cuáles tienen una desviación de 0
columnas_constantes = np.where(stds_por_columna == 0)[0]

if len(columnas_constantes) > 0:
    print(f"¡Cuidado! Hay varianza 0 en las columnas: ")
    # Las que salgan son fruto del OneHotEncoding, son 0 todos los valores que toman en realidad, no nos interesan pues no causan diferencias entre grupos
    print(X_train_reg.columns[columnas_constantes])


¡Cuidado! Hay varianza 0 en las columnas: 
Index(['modo_solicitud_cambio_de_institucion_curso_internacional',
       'cualificacion_previa_educacion_superior_doctorado',
       'nacionalidad_colombiana', 'nacionalidad_moldava',
       'nacionalidad_mozambiquena', 'cualificacion_madre_7o_ano',
       'cualificacion_madre_curso_general_de_comercio',
       'cualificacion_padre_11o_ano_no_completado',
       'cualificacion_padre_2o_ano_complementario_de_secundaria',
       'cualificacion_padre_2o_ciclo_del_bachillerato_general',
       'cualificacion_padre_9o_ano_no_completado',
       'cualificacion_padre_contabilidad_y_administracion_complementaria',
       'cualificacion_padre_curso_complementario_de_secundaria',
       'cualificacion_padre_curso_general_de_comercio',
       'cualificacion_padre_curso_superior_especializado',
       'cualificacion_padre_curso_tecnico_profesional',
       'cualificacion_padre_frecuencia_de_educacion_superior',
       'cualificacion_padre_no_sabe_leer_ni

In [13]:
# Para solucionarlo:
# 1. Extraemos los nombres exactos de las columnas con varianza cero
columnas_a_eliminar = X_train_reg.columns[columnas_constantes]

# 2. Las eliminamos tanto del conjunto de entrenamiento como del de prueba
X_train_reg = X_train_reg.drop(columns=columnas_a_eliminar)
# Usamos errors='ignore' por si alguna de esas columnas no estuviera en el test
X_test_reg = X_test_reg.drop(columns=columnas_a_eliminar, errors='ignore') 

print(f"Se han eliminado {len(columnas_a_eliminar)} columnas constantes.")

# 3. Volvemos a instanciar y aplicar el escalador sobre los datos limpios
scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

# 4. Comprobamos de nuevo
print(f"\nRe-escalado completado.")
print(f"Media de X_train_scaled: {X_train_reg_scaled.mean():.4f}")
print(f"Std de X_train_scaled:   {X_train_reg_scaled.std():.4f}")

#Las volvemos a guardar
features_reg = X_train_reg.columns.tolist()
print(f"Features finales para regresión: {len(features_reg)}")

Se han eliminado 24 columnas constantes.

Re-escalado completado.
Media de X_train_scaled: 0.0000
Std de X_train_scaled:   1.0000
Features finales para regresión: 208


In [14]:
print(f"Train regresión: {X_train_reg.shape[0]} muestras")
print(f"Test regresión:  {X_test_reg.shape[0]} muestras")
print(f"\nEstadísticas del target (nota_media_2sem):")
print(y_reg.describe().round(2))

Train regresión: 2843 muestras
Test regresión:  711 muestras

Estadísticas del target (nota_media_2sem):
count    3554.00
mean       12.73
std         1.38
min        10.00
25%        11.71
50%        12.67
75%        13.67
max        18.57
Name: nota_media_2sem, dtype: float64


## 2.5 Guardamos los datasets preprocesados

In [15]:
import os
import pickle 
os.makedirs('datos_procesados', exist_ok=True)

# Clasificación
np.save('datos_procesados/X_train_clf.npy', X_train_clf)
np.save('datos_procesados/X_test_clf.npy', X_test_clf)
np.save('datos_procesados/y_train_clf.npy', y_train_clf)
np.save('datos_procesados/y_test_clf.npy', y_test_clf)
np.save('datos_procesados/X_train_clf_scaled.npy', X_train_clf_scaled)
np.save('datos_procesados/X_test_clf_scaled.npy', X_test_clf_scaled)

# Regresión
np.save('datos_procesados/X_train_reg.npy', X_train_reg)
np.save('datos_procesados/X_test_reg.npy', X_test_reg)
np.save('datos_procesados/y_train_reg.npy', y_train_reg)
np.save('datos_procesados/y_test_reg.npy', y_test_reg)
np.save('datos_procesados/X_train_reg_scaled.npy', X_train_reg_scaled)
np.save('datos_procesados/X_test_reg_scaled.npy', X_test_reg_scaled)

# Guardamos los scalers y encoders para usarlos después
with open('datos_procesados/scaler_clf.pkl', 'wb') as f:
    pickle.dump(scaler_clf, f)
with open('datos_procesados/scaler_reg.pkl', 'wb') as f:
    pickle.dump(scaler_reg, f)
with open('datos_procesados/label_encoder_obj.pkl', 'wb') as f:
    pickle.dump(le_objetivo, f)

# Guardamos también los nombres de features
pd.Series(features_clf).to_csv('datos_procesados/features_clf.csv', index=False)
pd.Series(features_reg).to_csv('datos_procesados/features_reg.csv', index=False)

print("Todos los datos preprocesados guardados en 'datos_procesados/'")


Todos los datos preprocesados guardados en 'datos_procesados/'


## Resumen del preprocesado

| Aspecto | Decisión |
|---|---|
| Variables nulas | No hay, no se aplica imputación |
| Categóricas | LabelEncoder para bin, OHE para el resto |
| Escalado post split | StandardScaler (solo para modelos sensibles a escala) |
| Split | 80/20 estratificado (clasificación), 80/20 aleatorio (regresión) |
| Clases clasificación | Todas excepto eliminadas por ser 0 |
| Clases regresión | Eliminadas 5 variables del 2do semestre + objetivo |
| Filtro regresión | Solo estudiantes con nota_media_2sem > 0 |
